# GSM8K + GSM-IC motivation run — unguided / symbolic / semantic
G0 passed (PROCEED). This runs the three arms across context buckets and renders the two-panel
figure (Panel A: accuracy gap; Panel B: semantic latency grows, symbolic flat). Symbolic uses the
**format-only** calculation grammar (static, compiled once). Semantic uses the MathChecker
(provenance + exact arithmetic; no embedder, no tau). Resumable to Drive. See
`docs/gsm8k_motivation_plan.md`.

## Cell 1 — Setup (clone gsm8k branch, deps, Drive, HF login)

In [ ]:

SELF_REPO   = "https://github.com/knx4dmn/dag-motivation-exp"
SELF_BRANCH = "gsm8k"
import subprocess, sys, os, torch
torch_ver = torch.__version__.split("+")[0]
open("/content/constraints.txt", "w").write(f"torch=={torch_ver}\n")
if not os.path.isdir("/content/dag-motivation-exp"):
    subprocess.run(["git", "clone", "-b", SELF_BRANCH, SELF_REPO, "/content/dag-motivation-exp"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/constraints.txt",
                "-e", "/content/dag-motivation-exp[colab]"], check=True)
from google.colab import drive; drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/gsm8k_motivation"; os.makedirs(OUT_DIR, exist_ok=True)
from huggingface_hub import login; login()
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| out:", OUT_DIR)

## Cell 2 — Load model + tokenizer, resolve stop token

In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from motivation_exp import config as C
assert "T4" in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
tok = AutoTokenizer.from_pretrained(C.PRIMARY_MODEL)
model = AutoModelForCausalLM.from_pretrained(C.PRIMARY_MODEL, torch_dtype=torch.float16,
                                             attn_implementation="sdpa", device_map="cuda").eval()
EOT_ID = tok.convert_tokens_to_ids(C.STOP_TOKEN)
assert tok.convert_ids_to_tokens(EOT_ID) == C.STOP_TOKEN
MODEL_SHORT = C.MODEL_SHORT[C.PRIMARY_MODEL]
print("loaded", C.PRIMARY_MODEL, "| eot_id:", EOT_ID)

## Cell 3 — Data: GSM8K test, bucket to context targets
Pilot uses 20 items/bucket over [512, 1024, 2048]; raise `BUCKETS`/`N_PER_BUCKET` for the full run.
Seeded template distractors + validation gate (exact-rational gold chain survives injection).

In [ ]:

from motivation_exp import gsm8k_datagen as dg
BUCKETS = [512, 1024, 2048]          # full run: [512, 1024, 2048, 4096, 8192]
N_PER_BUCKET = 20                    # full run: 100
N_BASE = N_PER_BUCKET * 3 + 24       # spares for validation drops
count_tokens = dg.hf_token_counter(tok)
bases = dg.load_gsm8k(split="test", limit=N_BASE)     # GSM8K TEST split
raw = dg.bucket_gsm_to_targets(bases, count_tokens, BUCKETS, base_seed=0, log=print)
buckets = {}
for b in BUCKETS:
    kept = [it for it in raw[b] if dg.validate_gsm_item(it, count_tokens)[0]][:N_PER_BUCKET]
    buckets[b] = kept
    print(f"bucket {b}: {len(kept)} valid, mean tokens {sum(i.token_count for i in kept)/max(1,len(kept)):.0f}")

## Cell 4 — Run three arms (resumable to Drive)
Interleaves unguided/symbolic/semantic per item; skips any (model,method,bucket,item) already in the
JSONL. Symbolic grammar compiled once. Safe to re-run after a disconnect.

In [ ]:

import os
from motivation_exp import gsm8k_runner as R, grammar as gr
from motivation_exp.runner import DecodeCfg, make_torch_helpers
from motivation_exp.math_checker import MathChecker
from motivation_exp import config as C

sync, sample_fn = make_torch_helpers()
compiler = gr.make_compiler(tok, model.config.vocab_size)     # shared XGrammar compiler
cfg = DecodeCfg(eot_id=EOT_ID, max_new_tokens=C.MAX_NEW_TOKENS, per_step_cap=C.PER_STEP_TOKEN_CAP,
                temp_schedule=tuple(C.TEMPERATURE_SCHEDULE), max_retries=C.SEMANTIC_MAX_RETRIES,
                prefill_chunk_size=C.PREFILL_CHUNK, deterministic_decode=True)
OUT = os.path.join(OUT_DIR, "results.jsonl")
R.run_all(model, tok, buckets, C.METHODS, OUT, MODEL_SHORT, cfg=cfg, sync=sync, sample_fn=sample_fn,
          compiler=compiler, checker_factory=lambda: MathChecker(),
          gpu=torch.cuda.get_device_name(0))
print("done ->", OUT)

## Cell 5 — Gates + two-panel figure
G1: unguided acc at the smallest bucket in [0.40, 0.80] (sanity). G2: semantic >= symbolic and the
gap does not shrink with context. G3: semantic per-token latency rises, symbolic ~flat.

In [ ]:

from motivation_exp import plots
df = plots.load_results(OUT)
accT = plots._accuracy_table(df); latT = plots._latency_table(df)
bmin = min(BUCKETS); bmax = max(BUCKETS)
g1 = 0.40 <= accT.get("unguided", {}).get(bmin, (0,))[0] <= 0.80
def gap(b): return accT.get("semantic", {}).get(b, (0,))[0] - accT.get("symbolic", {}).get(b, (0,))[0]
g2 = gap(bmax) >= gap(bmin) - 1e-9 and all(accT["semantic"][b][0] >= accT["symbolic"][b][0] - 1e-9 for b in BUCKETS if b in accT.get("semantic",{}) and b in accT.get("symbolic",{}))
def lat(m,b): return latT.get(m, {}).get(b, (0,))[0]
g3 = lat("semantic", bmax) > lat("semantic", bmin) and abs(lat("symbolic", bmax) - lat("symbolic", bmin)) < 0.5 * max(1e-9, lat("symbolic", bmin))
for m in C.METHODS:
    print(m, "acc:", {b: round(accT.get(m,{}).get(b,(0,))[0],3) for b in BUCKETS},
             "| ms/tok:", {b: round(lat(m,b),1) for b in BUCKETS})
print(f"\nG1 unguided-sanity: {'PASS' if g1 else 'FAIL'} | G2 acc-gap: {'PASS' if g2 else 'FAIL'} | G3 latency: {'PASS' if g3 else 'FAIL'}")
png = plots.make_two_panel_figure(df, os.path.join(OUT_DIR, "motivation.png"), model_name=MODEL_SHORT)
print("figure ->", png)
from IPython.display import Image; Image(png)